# Lab 5: Tensor Manipulation with PyTorch

In [10]:
%pip install torch


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import torch
import time

The goal of this lab is to get familiar with basic tensor operations in PyTorch, understand the relationship between a tensor and its underlying storage, and appreciate the efficiency of tensor computations compared to their iterative Python equivalents.

---
## Exercise 1: Building a Matrix with Slicing

In this exercise, you will generate a structured matrix without using any Python loop — only tensor operations and slicing.

### Steps:
1. Create a $13 \times 13$ tensor filled with ones.
2. Use slicing to set the appropriate entries to 2 and 3.
3. The result should be the following matrix:

```
1 2 1 1 1 1 2 1 1 1 1 2 1
2 2 2 2 2 2 2 2 2 2 2 2 2
1 2 1 1 1 1 2 1 1 1 1 2 1
1 2 1 3 3 1 2 1 3 3 1 2 1
1 2 1 3 3 1 2 1 3 3 1 2 1
1 2 1 1 1 1 2 1 1 1 1 2 1
2 2 2 2 2 2 2 2 2 2 2 2 2
1 2 1 1 1 1 2 1 1 1 1 2 1
1 2 1 3 3 1 2 1 3 3 1 2 1
1 2 1 3 3 1 2 1 3 3 1 2 1
1 2 1 1 1 1 2 1 1 1 1 2 1
2 2 2 2 2 2 2 2 2 2 2 2 2
1 2 1 1 1 1 2 1 1 1 1 2 1
```

**Hint:** Use `torch.full` and slicing with step (`::`).

In [12]:
# TODO: Create the 13x13 matrix using torch.full and slicing (no Python loops)
m = torch.full((13, 13), 1.0, dtype=torch.int8)

m[3:5, 3:5] = 3
m[8:10, 3:5] = 3
m[3:5, 8:10] = 3
m[8:10, 8:10] = 3

m[:, 1::5] = 2
m[1::5, :] = 2

print(m)

tensor([[1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1],
        [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
        [1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1],
        [1, 2, 1, 3, 3, 1, 2, 1, 3, 3, 1, 2, 1],
        [1, 2, 1, 3, 3, 1, 2, 1, 3, 3, 1, 2, 1],
        [1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1],
        [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
        [1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1],
        [1, 2, 1, 3, 3, 1, 2, 1, 3, 3, 1, 2, 1],
        [1, 2, 1, 3, 3, 1, 2, 1, 3, 3, 1, 2, 1],
        [1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1],
        [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
        [1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1]], dtype=torch.int8)


---
## Exercise 2: Matrix Multiplication Throughput

Measure how fast PyTorch can perform large matrix multiplications.

### Steps:
1. Generate two $5000 \times 5000$ matrices filled with random Gaussian coefficients.
2. Compute their product and measure the time required.
3. Estimate the number of floating-point multiplications performed per second (should be in the order of billions).

**Hint:** Use `torch.empty`, `.normal_()`, `torch.mm` (or the `@` operator), and `time.perf_counter`.

In [13]:
# TODO: Generate two 5000x5000 Gaussian matrices, multiply them, and measure the throughput
d = 5000
a = torch.empty(d, d).normal_()
b = torch.empty(d, d).normal_()

time1 = time.perf_counter()
c = a @ b
time2 = time.perf_counter()

print('Throughput {:e} flop/s'.format((d * d * d) / (time2 - time1)))

Throughput 6.865551e+11 flop/s


---
## Exercise 3: Loops vs. Tensor Operations

Compare a naive Python loop implementation with a vectorized tensor operation to see the performance difference.

### Steps:
1. Write a function `mul_row` **using Python loops** (no slicing) that takes a 2D tensor and returns a tensor of the same size where the first row is unchanged, the second row is multiplied by 2, the third by 3, etc.

**Example:**

```python
>>> m = torch.full((4, 8), 2.0)
>>> m
tensor([[2., 2., 2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2., 2., 2.]])
>>> mul_row(m)
tensor([[2., 2., 2., 2., 2., 2., 2., 2.],
        [4., 4., 4., 4., 4., 4., 4., 4.],
        [6., 6., 6., 6., 6., 6., 6., 6.],
        [8., 8., 8., 8., 8., 8., 8., 8.]])
```

2. Write a second version `mul_row_fast` using **tensor operations** (no loops).
3. Apply both to a $1000 \times 400$ matrix and measure the time for each. There should be more than two orders of magnitude of difference.

**Hint:** For the fast version, use broadcasting with `torch.arange` and `torch.view`.

In [14]:
# Slow version: Python loops
def mul_row(m):
    r = torch.empty(m.size())
    for i in range(m.size(0)):
        for j in range(m.size(1)):
            r[i, j] = m[i, j] * (i + 1)
    return r

# TODO: Write mul_row_fast using tensor operations (no loops)
def mul_row_fast(m):
    c = torch.arange(1, m.size(0) + 1).view(-1, 1).float()
    return m * c

# Benchmark
m = torch.empty(1000, 400).normal_()

time1 = time.perf_counter()
a = mul_row(m)
time2 = time.perf_counter()
b = mul_row_fast(m)
time3 = time.perf_counter()

print('Loop version:   {:.4f}s'.format(time2 - time1))
print('Tensor version: {:.6f}s'.format(time3 - time2))
print('Speed ratio: {:.0f}x'.format((time2 - time1) / (time3 - time2)))
print('Sanity check: error is {:.6f}'.format(torch.norm(a - b).item()))

Loop version:   1.0600s
Tensor version: 0.000447s
Speed ratio: 2369x
Sanity check: error is 0.000000


**Question 1:** Why is the tensor version so much faster than the loop version? What is happening under the hood?

It parallelises the computation.

---
## Exercise 4: Nearest Neighbour Classification

Write a function that, given a training set and test samples, returns the label of the nearest training point for each test sample.

### Steps:
1. Write a function `nearest_classification(train_input, train_target, x)` where:
   - `train_input` is a 2D float tensor of shape $n \times d$ (training vectors),
   - `train_target` is a 1D long tensor of shape $n$ (training labels),
   - `x` is a 2D float tensor of shape $m \times d$ (test vectors),
   - The function returns the label of the nearest training example (using $L^2$ norm) for each test vector in `x`.

2. The function should **not** use any Python loop.

**Hint:** Use `torch.cdist` and `torch.argmin`.

In [15]:
# TODO: Implement nearest_classification without any Python loop
def nearest_classification(train_input, train_target, x):
    distances = torch.cdist(x, train_input) ** 2  # (m, n)
    nearest_indices = torch.argmin(distances, dim=1)  # (m,)
    return train_target[nearest_indices]  # (m,)

---
## Exercise 5: Error Computation with Optional Projection

Write a more general function that computes the number of classification errors using the 1-nearest-neighbour rule, with optional centering and dimensionality reduction.

### Steps:
1. Write a function:

```python
def compute_nb_errors(train_input, train_target, test_input, test_target, mean=None, proj=None):
```

where:

- `train_input` is a 2D float tensor of shape $n \times d$ (training vectors),
- `train_target` is a 1D long tensor of shape $n$ (training labels),
- `test_input` is a 2D float tensor of shape $m \times d$ (test vectors),
- `test_target` is a 1D long tensor of shape $m$ (test labels),
- `mean` is either `None` or a 1D float tensor of shape $d$,
- `proj` is either `None` or a 2D float tensor of shape $c \times d$.

2. The function subtracts `mean` (if not `None`) from both `train_input` and `test_input`, applies the projection `proj` (if not `None`) to both, and returns the number of classification errors using the 1-nearest-neighbour rule on the resulting data.

In [16]:
# TODO: Implement compute_nb_errors
def compute_nb_errors(train_input, train_target, test_input, test_target, mean=None, proj=None):
    if mean is not None:
        train_input = train_input - mean
        test_input = test_input - mean

    if proj is not None:
        train_input = train_input @ proj.t()
        test_input = test_input @ proj.t()

    # Predict labels for all test inputs
    predicted_labels = nearest_classification(train_input, train_target, test_input)

    # Count misclassifications
    nb_errors = (predicted_labels != test_target).sum().item()

    return nb_errors

---
## Exercise 6: Random Projection Experiment

Compare the performance of the 1-nearest-neighbour rule on raw data versus data projected onto a random subspace.

### Steps:
1. Load the breast cancer dataset from scikit-learn.
2. Split it into training and test sets.
3. Compute the error rate using the 1-nearest-neighbour rule on the **raw** data (baseline).
4. Generate a random Gaussian projection matrix of shape $10 \times d$ and compute the error rate on the **projected** data.
5. Compare the two error rates.

In [17]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# 1. Load data
X, y = load_breast_cancer(return_X_y=True)
print(f"Dataset shape: {X.shape}")

Dataset shape: (569, 30)


In [18]:
# 2. Split into train and test sets and convert to tensors
train_input, test_input, train_target, test_target = train_test_split(X, y, test_size=0.2)
train_input = torch.tensor(train_input, dtype=torch.float32)
train_target = torch.tensor(train_target, dtype=torch.long)
test_input = torch.tensor(test_input, dtype=torch.float32)
test_target = torch.tensor(test_target, dtype=torch.long)

# 3. Baseline: 1-NN on raw data
nb_errors = compute_nb_errors(train_input, train_target, test_input, test_target)
print('Baseline — errors: {:d}, error rate: {:.02f}%'.format(nb_errors, 100 * nb_errors / test_input.size(0)))

# 4. Random projection to 10 dimensions
basis = train_input.new(10, train_input.size(1)).normal_()

nb_errors = compute_nb_errors(train_input, train_target, test_input, test_target, None, basis)
print('Random {:d}d projection — errors: {:d}, error rate: {:.02f}%'.format(
    basis.size(0), nb_errors, 100 * nb_errors / test_input.size(0)))

Baseline — errors: 9, error rate: 7.89%
Random 10d projection — errors: 7, error rate: 6.14%


**Question 2:** How does the random projection affect the error rate? Why might projecting to a lower-dimensional space still preserve classification accuracy? (Look up the [Johnson-Lindenstrauss lemma](https://en.wikipedia.org/wiki/Johnson%E2%80%93Lindenstrauss_lemma).)

*Write your answer here:*